# Turner angle — three definitions compared

**Def 1 (LEGACY formula — reconstructed inline):**
`Tu = arctan[ρ₀(β²|∇S|² − α²|∇T|²) / (−gradrho2/ρ₀)]`
(numerator from constant-α/β linear model; denominator the MEASURED
JMD95 gradrho2).  This was the store channel until 2026-08-06; it
no longer exists in src and is computed inline here to PRESERVE
this comparison.

**Def 2 (closed-set):** same numerator; denominator rebuilt from
the same T,S samples via the direct cross term:
`D = −ρ₀(α²|∇T|² + β²|∇S|² − 2αβ∇T·∇S)`.

**Def 3 (projection — Johnson et al. 2012; Whalen & Drushka 2025):**
`Tu = arctan[∇ρ·(α∇T + β∇S) / ∇ρ·(α∇T − β∇S)]`
with ∇ρ measured (full EOS).

**They are mathematically equivalent** under the exact linearized
EOS with constant α, β — substitute ∇ρ = ρ₀(β∇S − α∇T) into Def 3
and Defs 1–2 fall out.  They differ in practice because measured
∇ρ carries EOS nonlinearity and locally-varying α, β; away from
compensation that gap is a small correction, but AT compensation
the leading terms cancel and the gap dominates — the three forms
diverge precisely where Tu is interesting.  Def 3 is the
best-conditioned (measured ∇ρ enters LINEARLY in numerator and
denominator, so its errors partially cancel in the ratio) and is
the literature definition; positive angles = temperature-dominated
(their convention, arctan on −90..90°).  **Def 3 is the production
`turner_angle` channel since 2026-08-06** — its column here calls
`calculate_fields.turner_angle` directly.

All dot products here use the co-located staggered-product form
(interp AFTER multiplying).  Undefined cores (∇ρ → 0) masked below
a visible floor.

## Section 1 — Grid

All three definitions are computed LIVE (Def 1 no longer exists in
src; the store now carries Def 3) — only the stitched-grid
coordinates are needed.

In [ ]:
# Section 1: shared 2D grid (XC/YC) for stitching + slicing.
import numpy as np
import matplotlib.pyplot as plt
import cmocean.cm as cmo

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
PIPELINE = "SURF"
DATE = "2012-11-09 12:00:00"

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket="dbof", folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat
print(f"grid: XC {XC.shape}")

## Section 2 — Region selection

In [ ]:
# Section 2: pick the region; re-run from here after changing it.
from dbof.plotting import regions

REGION       = "gulf_stream"   # <-- change me
ZOOM_HALF_KM = 100.0           # 200x200 km zoom box

_zoomable = [n for n, r in regions.REGIONS.items() if "zoom" in r]
assert REGION in _zoomable, f"pick one of {_zoomable}"
print(f"region: {REGION}  (options: {_zoomable})")

## Section 3 — All three definitions, live

Def 1 is the LEGACY formula, reconstructed inline (removed from src
2026-08-06).  Def 2 uses the production
`ng.calculate_grad_dot_tracer`; Def 3 IS the production
`calculate_fields.turner_angle`.

In [ ]:
# Section 3a: all three definitions, lazy.
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.preprocessing.physical_constants import (
    ALPHA, BETA, RHO0_REFERENCE,
)

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}")

T, S = ds_merge.Theta, ds_merge.Salt

# -- Def 1 (LEGACY: pre-2026-08-06 store channel, inline) ---------
# Numerator: constant-alpha/beta linear model; denominator: the
# MEASURED sq-first gradrho2.  Preserved here for the record.
gt = calculate_fields.grad_theta2(ds_merge, xgrid)
gs = calculate_fields.grad_salt2(ds_merge, xgrid)
gr = calculate_fields.grad_rho2(ds_merge, xgrid)
tu_def1 = np.degrees(np.arctan(
    (RHO0_REFERENCE * (BETA**2 * gs - ALPHA**2 * gt))
    / (-(gr.where(gr > 0)) / RHO0_REFERENCE)))

# -- Def 2 (closed-set, constant alpha/beta linear model) ---------
A = ALPHA**2 * ng.calculate_grad_squared_tracer(T, ds_merge, xgrid)
B = BETA**2 * ng.calculate_grad_squared_tracer(S, ds_merge, xgrid)
X = ALPHA * BETA * ng.calculate_grad_dot_tracer(
    T, S, ds_merge, xgrid)
tu_closed = np.degrees(np.arctan(
    (RHO0_REFERENCE * (B - A))
    / (-RHO0_REFERENCE * (A + B - 2.0 * X))))
d_closed = np.abs(A + B - 2.0 * X)      # mask variable, Def 2

# -- Def 3 (projection) = THE PRODUCTION CHANNEL ------------------
rho = calculate_fields.potential_density(ds_merge)
tu_proj = calculate_fields.turner_angle(ds_merge, xgrid, rho=rho)
d_proj = ng.calculate_grad_dot_tracer(rho, rho, ds_merge,
                                      xgrid)   # |grad rho|^2

live_map = {"tu_def1": tu_def1,
            "tu_closed": tu_closed, "d_closed": d_closed,
            "tu_proj": tu_proj, "d_proj": d_proj}
print("lazy Defs 1-3 ready")

In [ ]:
# Section 3b: stitch + slice (shared plumbing).
from dbof.plotting.live_fields import stitch_and_slice

region_arrays = stitch_and_slice(
    live_map, ds_raw, ds_merge, XC, YC, [REGION], batch=2)
print("sliced:", sorted(region_arrays))

## Section 4 — Masks

Defs 2–3 are undefined where their denominators vanish;
`FLOOR_PCT` masks the weakest x % of each mask variable.  Def 1
is shown unmasked (as the legacy channel was).

In [ ]:
# Section 4: mask the undefined cores (regional, tunable).
FLOOR_PCT = 1.0    # mask the weakest x% — tune me


def _masked(tu_key, d_key):
    """Regionally mask a Tu variant below its floor.
    Inputs: tu_key, d_key (str).  Outputs: (x, y, arr).
    Generated by LH and Claude
    """
    x, y, tu = region_arrays[tu_key][REGION]
    _, _, d = region_arrays[d_key][REGION]
    d = np.abs(d)
    floor = np.nanpercentile(d[np.isfinite(d)], FLOOR_PCT)
    out = np.where(d > floor, tu, np.nan)
    frac = 100.0 * np.mean(~np.isfinite(out) & np.isfinite(tu))
    print(f"{tu_key}: floor P{FLOOR_PCT} -> masked {frac:.2f}%")
    return x, y, out


DEFS = {
    "Def 1 (legacy formula)": region_arrays["tu_def1"][REGION],
    "Def 2 (closed-set)": _masked("tu_closed", "d_closed"),
    "Def 3 (projection, W&D)": _masked("tu_proj", "d_proj"),
}

## Section 5 — Maps: columns = definitions, rows = full / zoom

Fixed ±90° balance scale everywhere.  Gray in Defs 2–3 = masked
undefined core.  Remember the equivalence caveat: differences
between the columns are the EOS-nonlinearity/local-α,β content,
concentrated near compensated water.

In [ ]:
# Section 5: 2x3 comparison figure.
from dbof.plotting.pipeline_grids import LAND_COLOR

fig, axes = plt.subplots(2, 3, figsize=(5.4 * 3, 3.6 * 2))
pm = None
for col, (label, (x, y, arr)) in enumerate(DEFS.items()):
    for row, zoom in enumerate((False, True)):
        xx, yy, aa = (regions.crop_zoom(x, y, arr, REGION,
                                        half_km=ZOOM_HALF_KM)
                      if zoom else (x, y, arr))
        ax = axes[row, col]
        ax.set_facecolor(LAND_COLOR)
        pm = ax.pcolormesh(xx, yy, aa, cmap=cmo.balance,
                           vmin=-90, vmax=90, shading="nearest")
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel("zoom" if zoom else "full", fontsize=10)
    axes[0, col].set_title(label, fontsize=11)
fig.colorbar(pm, ax=list(axes.ravel()), orientation="horizontal",
             shrink=0.6, pad=0.03, label="Turner angle (deg)")
fig.suptitle(f"turner_angle \u2014 three definitions, {REGION}",
             fontsize=13)
plt.show()

## Section 6 — Distributions

One line per definition.  The speckle signature is a spurious ±90°
pileup; the definitions' magnitude offsets show up as shifted bulk
structure.

In [ ]:
# Section 6: Tu histograms, all three definitions.
bins = np.linspace(-90, 90, 181)
fig, ax = plt.subplots(figsize=(9, 4))
for (label, (_, _, arr)), colr in zip(
        DEFS.items(), ("firebrick", "seagreen", "steelblue")):
    v = arr[np.isfinite(arr)]
    ax.hist(v, bins=bins, histtype="step", density=True,
            label=f"{label}  (n={v.size})", color=colr)
ax.set_xlabel("Turner angle (deg)")
ax.set_ylabel("density")
ax.axvline(45, color="gray", lw=0.5)
ax.axvline(-45, color="gray", lw=0.5)
ax.legend()
ax.set_title(f"Tu distributions, {REGION}")
plt.show()

## Findings

**Decision: DEF 3** — adopted as the production `turner_angle`
channel (2026-08-06), computed LITERALLY: ∇ρ is the MEASURED
density gradient (JMD95, full EOS) and every dot product uses the
co-located staggered-product form (`ng.calculate_grad_dot_tracer` —
products formed where their factors natively live, moved to the
centre afterwards).  This is the definition used by Whalen &
Drushka (2025), following Johnson et al. (2012).

Under the exact constant-α/β linearized EOS this collapses to the
previous magnitudes-based form
arctan[ρ₀(β²|∇S|² − α²|∇T|²)/(−|∇ρ|²/ρ₀)]; the two diverge where
EOS nonlinearity / local α,β variation matters — enriched in
weak-|∇ρ|² water (A/B above; plan 2026-08-06: corr 0.89, median
|diff| 0.5°, 61% of the >10° disagreement in the weakest quartile
of |∇ρ|²).  Positive angles = temperature-dominated density
gradients; ±45° = single-tracer fronts; near ±90° =
density-compensated.

NOTE: no weak-gradient mask is applied in the store — it keeps the
raw definition (arctan is bounded; exact 0/0 pixels yield NaN
naturally).  Tu is low-confidence where |∇ρ|² is small regardless
of definition; mask at analysis/display time (`FLOOR_PCT` above).